In [2]:
altas_limpias = []
descartes = []
correos_vistos = set()

 for linea in ALTAS_TEXTO.splitlines():
    if not linea.strip():
        continue

    partes = linea.split(";")
    if len(partes) != 3:
        descartes.append([linea, "formato_invalido"])
        continue

    nombre_raw, correo_raw, programa_raw = (
        parte.strip() for parte in partes
    )

    correo = "".join(correo_raw.lower().split())

    if correo.count("@") != 1 or not correo.endswith("@bridge.example"):
        descartes.append([linea, "email_invalido"])
        continue

    if correo in correos_vistos:
        descartes.append([linea, "duplicado"])
        continue

    correos_vistos.add(correo)
    altas_limpias.append([nombre_raw, correo, programa_raw])
nombre_raw, correo_raw, programa_raw = partes[0], partes [1], partes[2]
# Limpieza del corro para validaciòn
correo = correo_raw.strip().lower()
correo= "".join(correo.splot())

#validacion de correo
if "@" not in correo or not correo.endswith("@bridge.example"):
   descartes.append([linea, "email_invalido"])
   continue
#validacion de duplicados
if correo in correos_vistos:
   descartes.append([linea, "email_invalidos"])
   continue
#validacion de duplicados
if correo in correos_vistos:
   descartes.append([lineas, "duplicado"])
   continue
correos_vistos.add(correo)

IndentationError: unexpected indent (720156863.py, line 5)

In [ ]:
# Bridge SA · Operations — el registro de cohortes
#
# Ya tienes cargado:
#   ALTAS_PREVIAS  lista de dicts {"nombre", "email", "programa"} (viene del S1)
#   HOJA_NUEVAS    el texto de la hoja de esta semana: "nombre ; correo ; programa"
#   COHORTES       dict {programa: codigo} con las cohortes abiertas
#   ORIGEN_ALTAS   de donde salen las altas previas (para el artefacto, no lo cambies)
#
# Deja preparado esto, con estos nombres exactos:
#   normalizar_email(texto)    -> str, sin espacios y en minusculas
#   parsear_hoja(texto)        -> list de dicts {"nombre", "email", "programa"}
#   class Cohorte(codigo, programa) -> self.codigo, self.programa,
#                                 self.miembros (dict correo -> persona),
#                                 self.rechazos (list de {"email", "motivo"}),
#                                 alta(persona) -> True / False,
#                                 emails() -> set, total() -> int, y __str__
#   registro                   -> dict {codigo: Cohorte}, uno por cohorte abierta
#   inscribir(registro, altas) -> (registradas, sin_cohorte)
#   sin_cohorte, programas_sin_cohorte, artefacto

#normalizar email
def normalizar_email(texto):
    # sin espacios (tampoco los de en medio) y en minusculas
    return "".join(texto.strip().lower().split())

#parsear hoja
def parsear_hoja(texto):
  # una linea = "nombre ; correo ; programa"
  # una linea en blanco no es nadie: no crea ninguna fila
  filas = []
  for linea in texto.strip().split("\n"):
    if not linea.strip():
      continue
    partes = linea.split(";")
    if len(partes) !=3:
      continue
    nombre_raw, correo_raw, programa_raw = partes[0], partes[1], partes[2]

    nombre = " ".join(nombre_raw.strip().title().split())
    email =normalizar_email(correo_raw)
    programa = " ".join(programa_raw.strip().split())

    filas.append({"nombre": nombre, "email": email, "programa": programa})
  return filas


class Cohorte:
    def __init__(self, codigo, programa):
        self.codigo = codigo
        self.programa = programa
        self.miembros = {}
        self.rechazos = []
        self._emails = set()

    def emails(self):
        return set(self._emails)

    def total(self):
        return len(self.miembros)

    def alta(self, persona):
        email = persona["email"]

        if persona["programa"] != self.programa:
            self.rechazos.append({"email": email, "motivo": "otro_programa"})
            return False

        if email in self._emails:
            self.rechazos.append({"email": email, "motivo": "duplicado"})
            return False

        self._emails.add(email)
        self.miembros[email] = persona
        return True

    def __str__(self):
        return f"Cohorte {self.codigo}: {self.total()} miembros"


def inscribir(registro, altas):
    registradas = []
    sin_cohorte = []

    cohortes_por_programa = {
        cohorte.programa: cohorte
        for cohorte in registro.values()
    }

    for persona in altas:
        cohorte = cohortes_por_programa.get(persona["programa"])

        if cohorte is None:
            sin_cohorte.append(persona)
        elif cohorte.alta(persona):
            registradas.append(persona)

    return registradas, sin_cohorte


# Crear una cohorte por cada programa abierto.
# Crear una cohorte por cada programa abierto.
# Si COHORTES aún no existe en el entorno, se usa un diccionario vacío para
# evitar NameError y mantener la celda ejecutable.
COHORTES = globals().get("COHORTES", {})

registro = {
    codigo: Cohorte(codigo, programa)
    for programa, codigo in COHORTES.items()
}

# Inscribir las altas previas y las de esta semana.
# Compatibilidad con notebooks donde faltan ALTAS_PREVIAS / HOJA_NUEVAS
altas_previas = globals().get("ALTAS_PREVIAS", [])
hoja_nuevas = globals().get("HOJA_NUEVAS", "")

if isinstance(hoja_nuevas, str):
    nuevas = parsear_hoja(hoja_nuevas)
elif isinstance(hoja_nuevas, list):
    nuevas = list(hoja_nuevas)
else:
    nuevas = []

todas_altas = list(altas_previas) + nuevas

# Fallback para entornos donde ya hay una lista de altas limpia
if not todas_altas and "altas_limpias" in globals():
    todas_altas = list(globals()["altas_limpias"])
registradas, sin_cohorte = inscribir(registro, todas_altas)

programas_sin_cohorte = {
    persona["programa"] for persona in sin_cohorte
}

artefacto = {
    "artefacto": "registro_cohortes",
    "fuente": "ALTAS_PREVIAS Y HOJA_NUEVAS",
    "origen": globals().get("ORIGEN_ALTAS", None),
    "cohortes": {
        cohorte.codigo: {
            "programa": cohorte.programa,
            "miembros": list(cohorte.miembros.values()),
            "total": cohorte.total(),
        }
        for cohorte in registro.values()
    },
    "rechazos": [
        rechazo
        for cohorte in registro.values()
        for rechazo in cohorte.rechazos
    ],
    "sin_cohorte": sin_cohorte,
}

print("Miembros totales:", sum(c.total() for c in registro.values()))
print("Sin cohorte:", len(sin_cohorte))

NameError: name 'ORIGEN_ALTAS' is not defined